DRIVE MOUNT

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


INSTALL DEPENDENCIES

In [2]:
!pip install -U \
langchain \
langchain-community \
langchain-huggingface \
sentence-transformers \
faiss-cpu \
pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

CONFIGURE ENVIRONMENTS AND PATHS

In [3]:
import os

base_path = "/content/drive/MyDrive/LegalResearch"

print(os.listdir(base_path))

['data', 'vectorstore', 'notebooks', 'app']


LOAD LEGAL DOCUMENTS

In [4]:
import os

metadata_path = "/content/drive/MyDrive/LegalResearch/data/metadata"

print("Files in metadata folder:")
print(os.listdir(metadata_path))

Files in metadata folder:
['laws.csv', 'law_case_mapping.csv', 'cases.csv']


In [5]:
import os

base_path = "/content/drive/MyDrive/LegalResearch"

for root, dirs, files in os.walk(base_path):
    if files:
        print("\nFolder:", root)
        print("Files:", files)


Folder: /content/drive/MyDrive/LegalResearch/data/laws
Files: ['hindu_marriage_act.pdf', 'hindu_succession_act.pdf', 'transfer_property_act.pdf', 'specific_relief_act.pdf', 'contract_labour_act.pdf', 'BSA.pdf', 'dpdp.pdf', 'posh_act.pdf', 'Bns.pdf', 'companies_act.pdf', 'competition_act.pdf', 'consumer_protection_act.pdf', 'it_act.pdf', 'pwdva.pdf', 'special_marriage_act.pdf', 'constitution.pdf', 'ibc.pdf', 'rera.pdf', 'the_registration_act.pdf']

Folder: /content/drive/MyDrive/LegalResearch/data/metadata
Files: ['laws.csv', 'law_case_mapping.csv', 'cases.csv']

Folder: /content/drive/MyDrive/LegalResearch/vectorstore/laws_faiss
Files: ['index.pkl', 'index.faiss']

Folder: /content/drive/MyDrive/LegalResearch/notebooks
Files: ['ingestion.ipynb', '03_rag_pipeline.ipynb.ipynb', '02_retrieval_testing.ipynb', '02_Legal_Chunking.ipynb', '01_build_vectorstore.ipynb']


In [6]:
import pandas as pd

laws_df = pd.read_csv(
    "/content/drive/MyDrive/LegalResearch/data/metadata/laws.csv"
)

cases_df = pd.read_csv(
    "/content/drive/MyDrive/LegalResearch/data/metadata/cases.csv"
)

mapping_df = pd.read_csv(
    "/content/drive/MyDrive/LegalResearch/data/metadata/law_case_mapping.csv"
)

In [7]:
print(os.listdir("/content/drive/MyDrive/LegalResearch/data"))

['laws', 'cases', 'metadata']


In [8]:
print(os.listdir("/content/drive/MyDrive/LegalResearch/data/laws")[:5])
print(os.listdir("/content/drive/MyDrive/LegalResearch/data/cases")[:5])

['hindu_marriage_act.pdf', 'hindu_succession_act.pdf', 'transfer_property_act.pdf', 'specific_relief_act.pdf', 'contract_labour_act.pdf']
[]


In [9]:
from langchain_community.document_loaders import PyPDFLoader
import os


BASE_PATH = "/content/drive/MyDrive/LegalResearch"

LAW_PATH = os.path.join(BASE_PATH, "data", "laws")
CASE_PATH = os.path.join(BASE_PATH, "data", "cases")


def load_documents(folder):

    docs = []

    print("\nLoading from:", folder)

    for file in os.listdir(folder):

        if file.lower().endswith(".pdf"):

            path = os.path.join(folder, file)

            loader = PyPDFLoader(path)

            pdf_docs = loader.load()

            docs.extend(pdf_docs)

            print(file, "->", len(pdf_docs), "pages")

    return docs



laws = load_documents(LAW_PATH)

cases = load_documents(CASE_PATH)


documents = laws + cases

print("\nTotal pages loaded:", len(documents))

/tmp/ipykernel_754/617214098.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader



Loading from: /content/drive/MyDrive/LegalResearch/data/laws
hindu_marriage_act.pdf -> 17 pages
hindu_succession_act.pdf -> 15 pages
transfer_property_act.pdf -> 2 pages
specific_relief_act.pdf -> 17 pages
contract_labour_act.pdf -> 22 pages
BSA.pdf -> 54 pages
dpdp.pdf -> 25 pages
posh_act.pdf -> 14 pages
Bns.pdf -> 102 pages
companies_act.pdf -> 370 pages
competition_act.pdf -> 46 pages
consumer_protection_act.pdf -> 40 pages
it_act.pdf -> 36 pages
pwdva.pdf -> 12 pages
special_marriage_act.pdf -> 25 pages
constitution.pdf -> 256 pages
ibc.pdf -> 142 pages
rera.pdf -> 36 pages
the_registration_act.pdf -> 24 pages

Loading from: /content/drive/MyDrive/LegalResearch/data/cases

Total pages loaded: 1255


EXTRACT AND INSPECT DOC CONTENT


In [10]:
print(documents[0].page_content[:500])
print(documents[0].metadata)

1 
 
 
 
 
 
 
 
 
 
 
The Hindu Marriage Act, 1955 
(ACT NO. 25 OF 1955) 
[As on the 15th April, 2026]
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-05-06T16:22:42+05:30', 'author': 'HP', 'moddate': '2026-05-06T16:22:42+05:30', 'source': '/content/drive/MyDrive/LegalResearch/data/laws/hindu_marriage_act.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}


In [11]:
for doc in documents:
    doc.metadata["source"] = os.path.basename(doc.metadata["source"])

print(documents[0].metadata)

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-05-06T16:22:42+05:30', 'author': 'HP', 'moddate': '2026-05-06T16:22:42+05:30', 'source': 'hindu_marriage_act.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}


In [12]:
print("Laws:", laws_df.shape)
print("Cases:", cases_df.shape)
print("Mappings:", mapping_df.shape)

Laws: (20, 9)
Cases: (30, 7)
Mappings: (35, 4)


In [13]:
print("LAWS COLUMNS")
print(laws_df.columns.tolist())

print("\nCASES COLUMNS")
print(cases_df.columns.tolist())

print("\nMAPPING COLUMNS")
print(mapping_df.columns.tolist())

LAWS COLUMNS
['law_id', 'law_name', 'year', 'category', 'jurisdiction', 'file_name', 'primary_citation', 'summary', 'keywords']

CASES COLUMNS
['id', 'title', 'year', 'court', 'citation', 'topic', 'jurisdiction']

MAPPING COLUMNS
['law_id', 'case_id', 'legal_topic', 'relevance']


In [14]:
from langchain_community.document_loaders import PyPDFLoader
import os

laws_folder = "/content/drive/MyDrive/LegalResearch/data/laws"

pdf_files = [
    f for f in os.listdir(laws_folder)
    if f.endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDFs")
print(pdf_files[:5])

Found 19 PDFs
['hindu_marriage_act.pdf', 'hindu_succession_act.pdf', 'transfer_property_act.pdf', 'specific_relief_act.pdf', 'contract_labour_act.pdf']


In [15]:
from langchain_community.document_loaders import PyPDFLoader
import os

test_pdf = os.path.join(
    "/content/drive/MyDrive/LegalResearch/data/laws",
    "constitution.pdf"
)

loader = PyPDFLoader(test_pdf)

docs = loader.load()

print("Number of pages:", len(docs))

print("\n--- FIRST PAGE PREVIEW ---\n")

print(docs[0].page_content[:1000])

Number of pages: 256

--- FIRST PAGE PREVIEW ---

THE CONSTITUTION OF INDIA 
[As on 9th December, 2020] 
 
 
 
 
 
2020 
 
 
GOVERNMENT OF INDIA 
MINISTRY OF LAW AND JUSTICE 
LEGISLATIVE DEPARTMENT


In [16]:
print(docs[0].metadata)

{'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2020-12-09T20:13:14-08:00', 'author': 'Naseem', 'moddate': '2020-12-09T20:13:14-08:00', 'source': '/content/drive/MyDrive/LegalResearch/data/laws/constitution.pdf', 'total_pages': 256, 'page': 0, 'page_label': '1'}


In [17]:
from langchain_community.document_loaders import PyPDFLoader
import os

laws_folder = "/content/drive/MyDrive/LegalResearch/data/laws"

all_law_docs = []

for pdf_file in os.listdir(laws_folder):

    if pdf_file.endswith(".pdf"):

        pdf_path = os.path.join(
            laws_folder,
            pdf_file
        )

        loader = PyPDFLoader(pdf_path)

        docs = loader.load()

        all_law_docs.extend(docs)

        print(
            f"Loaded {pdf_file} : {len(docs)} pages"
        )

print("\nTotal pages loaded:", len(all_law_docs))

Loaded hindu_marriage_act.pdf : 17 pages
Loaded hindu_succession_act.pdf : 15 pages
Loaded transfer_property_act.pdf : 2 pages
Loaded specific_relief_act.pdf : 17 pages
Loaded contract_labour_act.pdf : 22 pages
Loaded BSA.pdf : 54 pages
Loaded dpdp.pdf : 25 pages
Loaded posh_act.pdf : 14 pages
Loaded Bns.pdf : 102 pages
Loaded companies_act.pdf : 370 pages
Loaded competition_act.pdf : 46 pages
Loaded consumer_protection_act.pdf : 40 pages
Loaded it_act.pdf : 36 pages
Loaded pwdva.pdf : 12 pages
Loaded special_marriage_act.pdf : 25 pages
Loaded constitution.pdf : 256 pages
Loaded ibc.pdf : 142 pages
Loaded rera.pdf : 36 pages
Loaded the_registration_act.pdf : 24 pages

Total pages loaded: 1255


In [18]:
print(all_law_docs[0].metadata)
print(all_law_docs[-1].metadata)

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-05-06T16:22:42+05:30', 'author': 'HP', 'moddate': '2026-05-06T16:22:42+05:30', 'source': '/content/drive/MyDrive/LegalResearch/data/laws/hindu_marriage_act.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}
{'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-12-19T11:46:35-08:00', 'author': 'Lenovo', 'moddate': '2018-12-19T11:46:35-08:00', 'source': '/content/drive/MyDrive/LegalResearch/data/laws/the_registration_act.pdf', 'total_pages': 24, 'page': 23, 'page_label': '24'}


In [19]:
law_lookup = {}

for _, row in laws_df.iterrows():

    law_lookup[row["file_name"].lower()] = {
        "law_id": row["law_id"],
        "law_name": row["law_name"],
        "year": row["year"],
        "category": row["category"],
        "jurisdiction": row["jurisdiction"],
        "primary_citation": row["primary_citation"]
    }

print("Laws indexed:", len(law_lookup))

Laws indexed: 20


In [20]:
import os

for doc in all_law_docs:

    filename = os.path.basename(
        doc.metadata["source"]
    ).lower()

    if filename in law_lookup:

        doc.metadata.update(
            law_lookup[filename]
        )

In [21]:
print(all_law_docs[0].metadata)

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-05-06T16:22:42+05:30', 'author': 'HP', 'moddate': '2026-05-06T16:22:42+05:30', 'source': '/content/drive/MyDrive/LegalResearch/data/laws/hindu_marriage_act.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'law_id': 'L011', 'law_name': 'Hindu Marriage Act', 'year': 1955, 'category': 'Family Law', 'jurisdiction': 'India', 'primary_citation': 'Act No. 25 of 1955'}


In [22]:
print(all_law_docs[0].metadata)
print(all_law_docs[500].metadata)

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-05-06T16:22:42+05:30', 'author': 'HP', 'moddate': '2026-05-06T16:22:42+05:30', 'source': '/content/drive/MyDrive/LegalResearch/data/laws/hindu_marriage_act.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'law_id': 'L011', 'law_name': 'Hindu Marriage Act', 'year': 1955, 'category': 'Family Law', 'jurisdiction': 'India', 'primary_citation': 'Act No. 25 of 1955'}
{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-11-07T11:12:04+05:30', 'author': 'Administrator', 'moddate': '2024-11-07T11:12:04+05:30', 'source': '/content/drive/MyDrive/LegalResearch/data/laws/companies_act.pdf', 'total_pages': 370, 'page': 232, 'page_label': '233', 'law_id': 'L007', 'law_name': 'Companies Act', 'year': 2013, 'category': 'Corporate Law', 'jurisdiction': 'India', 'primary_citation': 'Act No. 18 of 2013'}


In [23]:
!pip show langchain
!pip show langchain-community

Name: langchain
Version: 1.3.14
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 


SPLIT DOCS INTO CHUNKS

In [24]:
!pip install -q langchain-text-splitters

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250,
    separators=[
        "\n\n",
        "\n",
        " ",
        ""
    ]
)


chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 4670


GENERATE EMBEDDINGS

In [30]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

print("Using device:", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using device: cpu


LOAD SAVED VECTOR STORE

In [38]:
!pip install -q sentence-transformers

In [39]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

device = "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [40]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embedding_model)

print(vector_db.index.ntotal)

4670


In [41]:
import os

VECTOR_PATH = "/content/drive/MyDrive/LegalResearch/vectorstore"

os.makedirs(VECTOR_PATH, exist_ok=True)

vector_db.save_local(VECTOR_PATH)

print("✅ Vector Store Saved Successfully!")

✅ Vector Store Saved Successfully!


In [42]:
import os

print(os.listdir("/content/drive/MyDrive/LegalResearch/vectorstore"))

['laws_faiss', 'index.faiss', 'index.pkl']


In [43]:
from langchain_community.vectorstores import FAISS

loaded_db = FAISS.load_local(
    "/content/drive/MyDrive/LegalResearch/vectorstore",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("Vectors Loaded:", loaded_db.index.ntotal)

Vectors Loaded: 4670


In [50]:
results = loaded_db.similarity_search(
    "What are the grounds for divorce under the Hindu Marriage Act?",
    k=10
)

for i, doc in enumerate(results, 1):
    print("="*80)
    print(f"Result {i}")
    print("Source:", doc.metadata["source"])
    print("Page:", doc.metadata["page"] + 1)
    print(doc.page_content[:1200])

Result 1
Source: hindu_marriage_act.pdf
Page: 3
3 
 
THE HINDU MARRIAGE ACT, 1955 
________ 
ARRANGEMENT OF SECTIONS 
_________ 
PRELIMINARY 
SECTIONS 
1. Short title and extent. 
2. Application of Act. 
3. Definitions. 
4. Overriding effect of Act. 
HINDU MARRIAGES 
5. Conditions for a Hindu marriage. 
6. [Omitted.]. 
7. Ceremonies for a Hindu marriage. 
8. Registration of Hindu marriages. 
RESTITUTION OF CONJUGAL RIGHTS AND JUDICIAL SEPARATION 
9. Restitution of conjugal rights. 
10. Judicial separation. 
NULLITY OF MARRIAGE AND DIVORCE 
11. Void marriages. 
12. Voidable marriages. 
13. Divorce. 
13A. Alternate relief in divorce proceedings. 
13B. Divorce by mutual consent. 
14. No petition for divorce to be presented within one year of marriage. 
15. Divorced persons when may marry again. 
16. Legitimacy of children of void and voidable marriages. 
17. Punishment of bigamy. 
18. Punishment for contravention of certain other conditions for a Hindu marriage. 
JURISDICTION AND PROCEDUR